# Main Optotagging Analysis Notebook


Main notebook for analysis of cell-type distribution throughout the entire Allen Institute Visual Behavior Neuropixels Dataset. 

Uses imports from the visb_analysis package, which reuses code from the Allen Institute Visual Behavior Optotagging Tutorial notebook.

### Loads data from `results/` folder

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import yaml

from visb_analysis.plots import (
    plot_units_per_session,
    plot_population_psth,
    plot_psth_heatmaps,
    plot_region_distribution,
    plot_cre_fraction_by_region,
    plot_region_comparison,
)

In [ ]:
# Change to which configuration you're analyzing
config_type = "Sst"

In [ ]:
config_path = Path("../configs") / f"{config_type}.yaml"
with config_path.open("r") as f:
    config = yaml.safe_load(f)
results_dir = Path(config["results_dir"]) / config_type

In [ ]:
all_psths, all_unit_ids, all_units_meta = [], [], []

In [ ]:
time_bins = None
for npz_path in sorted(results_dir.glob("*.npz")):
    session_id = int(npz_path.stem)
    data = np.load(npz_path)
    meta = pd.read_csv(results_dir / f"{session_id}_units.csv", index_col=0)
    all_psths.append(data["psth_array"])
    all_unit_ids.append(data["unit_ids"])
    all_units_meta.append(meta)
    if time_bins is None:
        time_bins = data["time_bins"]

if not all_psths:
    raise FileNotFoundError(f"No .npz files found in {results_dir}")

### Analyze & Visualize results

In [ ]:
psth_trial_avg = np.concatenate(all_psths, axis=0)  # (total_units, n_bins) — already trial-averaged
meta_all = pd.concat(all_units_meta)
meta_all["optotagged"] = meta_all["optotagged"].fillna(False).astype(bool)

opto_mask     = meta_all["optotagged"].values
psth_opto     = psth_trial_avg[opto_mask]
psth_non_opto = psth_trial_avg[~opto_mask]
meta_opto     = meta_all[opto_mask]
meta_non_opto = meta_all[~opto_mask]

genotype = config["sessions"]["genotype"]

print(f"Config:               {config_type}  ({genotype})")
print(f"Sessions loaded:      {len(all_psths)}")
print(f"Total units:          {len(psth_trial_avg)}")
print(f"  Optotagged:         {opto_mask.sum()}")
print(f"  Non-optotagged:     {(~opto_mask).sum()}")
print(f"\nRegion breakdown (optotagged):")
print(meta_opto["structure_acronym"].value_counts())

In [ ]:
fig = plot_units_per_session(all_units_meta, title=f'{genotype} — units per session')

### Population average PSTH

In [ ]:
fig = plot_population_psth(
    psth_opto, psth_non_opto, time_bins,
    title=f'Population PSTH — {genotype} optotagged vs non-optotagged',
)

### Unit heatmap (sorted by response magnitude)

In [ ]:
fig = plot_psth_heatmaps(
    psth_opto, psth_non_opto, time_bins,
    suptitle=f'{genotype} — image change PSTH heatmaps',
    time_range=(-0.25, 0.75),
)

### Cre+ distribution across brain regions

In [ ]:
fig = plot_region_distribution(meta_all, genotype=genotype)

In [ ]:
fig = plot_cre_fraction_by_region(meta_all, genotype=genotype, min_units=10)

### Population PSTH by region (top N regions)

In [ ]:
fig = plot_region_comparison(
    psth_opto, psth_non_opto,
    meta_opto, meta_non_opto, time_bins,
    top_n=10,
)